# Task 1: Model Training and Optimization Pipeline
Use this notebook to perform your data preprocessing, hyperparameter tuning via Cross-Validation, and final evaluation on the test set.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import time
import optuna
import trackio
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

# Add any other imports you need here
from scipy.stats import randint
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Data Loading & Preprocessing
Load `train.csv` and `test.csv`. Convert string categorical variables to numeric.
**Required:** Save your label encoders/mappings because your Streamlit UI will need them later to prepare user inputs for inference!

In [ ]:
train_df = pd.read_csv('Dataset/train.csv')
test_df = pd.read_csv('Dataset/test.csv')

print(f"Train Shape: {train_df.shape}")
train_df.head()

Train Shape: (11128, 15)


,location,city,latitude,longitude,price,numBathrooms,numBalconies,isNegotiable,SecurityDeposit,Status,Size_ft²,BHK,rooms_num,property_type,verification_days
0,Thane West,Mumbai,19.216236,72.980614,27000,2,0,0,0,Unfurnished,530,1,2,Apartment,10.0
1,Thane West,Mumbai,19.271788,72.967789,13500,2,0,0,0,Unfurnished,650,1,1,Apartment,1460.0
2,Airoli,Mumbai,19.143419,72.997307,20000,1,0,0,0,Semi-Furnished,700,1,1,Apartment,730.0
3,Mahipalpur,Delhi,28.547979,77.130135,14000,1,0,0,0,Furnished,1200,0,1,Studio Apartment,150.0
4,Jasola,Delhi,28.529716,77.293137,24500,2,1,0,49000,Unfurnished,1000,1,2,Apartment,180.0


In [ ]:
# TODO: Implement your preprocessing here (use LabelEncoder or manual dictionaries)
# Ensure you keep all necessary features that will be shown on the UI dashboard.

categorical_cols = ['location', 'city', 'Status', 'property_type']

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([train_df[col], test_df[col]], axis=0)
    le.fit(combined)
    train_df[col] = le.transform(train_df[col])
    test_df[col] = le.transform(test_df[col])
    label_encoders[col] = le
    print(f"Encoded {col} with {len(le.classes_)} unique values.")

train_df.head()

Encoded location with 702 unique values.
Encoded city with 4 unique values.
Encoded Status with 3 unique values.
Encoded property_type with 6 unique values.


,location,city,latitude,longitude,price,numBathrooms,numBalconies,isNegotiable,SecurityDeposit,Status,Size_ft²,BHK,rooms_num,property_type,verification_days
0,607,2,19.216236,72.980614,27000,2,0,0,0,2,530,1,2,0,10.0
1,607,2,19.271788,72.967789,13500,2,0,0,0,2,650,1,1,0,1460.0
2,6,2,19.143419,72.997307,20000,1,0,0,0,1,700,1,1,0,730.0
3,327,0,28.547979,77.130135,14000,1,0,0,0,0,1200,0,1,3,150.0
4,223,0,28.529716,77.293137,24500,2,1,0,49000,2,1000,1,2,0,180.0


In [ ]:
# TODO: Separate predictors (X) and target (y: 'price')
target_col = 'price'
feature_cols = [c for c in train_df.columns if c != target_col]

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

## 2. Hyperparameter Tuning using Cross-Validation

**Strict Search Space:**
- `n_estimators`: 50 to 200
- `max_depth`: 10 to 30
- `min_samples_split`: 2 to 10

Implement Grid Search, Random Search, and Bayesian Optimization (using Optuna). Evaluate each using 5-fold cross-validation on `train_df`.

In [ ]:
rf = RandomForestRegressor(random_state=42)

# TODO: Initialize trackio project/experiment
trackio.init(project='UrbanNest-RentPrediction')
trackio.finish()

* Trackio project initialized: UrbanNest-RentPrediction
* Trackio metrics logged to: C:\Users\r4849\.cache\huggingface\trackio
* Created new run: dainty-sunset-0


* Run finished. Uploading logs to Trackio (please wait...)


In [ ]:
# TODO: 1. Grid Search Implementation
# Use trackio to log the method name, time taken, number of iterations, and best cross-validation score

grid_params = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [10, 15, 20, 25, 30],
    'min_samples_split': [2, 5, 10]
}

start_time = time.time()

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=grid_params,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)
grid_search.fit(X_train, y_train)

grid_time = time.time() - start_time

grid_best_params = grid_search.best_params_
grid_best_score = -grid_search.best_score_

print(f'Grid Search completed in {grid_time:.2f}s')
print(f'Best params: {grid_best_params}')
print(f'Best CV MAE: {grid_best_score:.4f}')


Fitting 5 folds for each of 60 candidates, totalling 300 fits


Grid Search completed in 391.21s
Best params: {'max_depth': 25, 'min_samples_split': 2, 'n_estimators': 200}
Best CV MAE: 13270.7582


In [ ]:
run_grid = trackio.init(project='UrbanNest-RentPrediction', name='GridSearch')
trackio.log({
    'method': 'GridSearch',
    'best_cv_mae': float(grid_best_score),
    'time_seconds': float(grid_time),
    'n_iterations': 60,
    'best_n_estimators': int(grid_best_params['n_estimators']),
    'best_max_depth': int(grid_best_params['max_depth']),
    'best_min_samples_split': int(grid_best_params['min_samples_split']),
})
trackio.finish()

* Created new run: brave-forest-1
* Run finished. Uploading logs to Trackio (please wait...)


c:\Users\r4849\AppData\Local\Programs\Python\Python313\Lib\site-packages\trackio\__init__.py:257: UserWarning: * Warning: resume='never' but a run 'GridSearch' already exists in project 'UrbanNest-RentPrediction'. Generating a new name and instead. If you want to resume this run, call init() with resume='must' or resume='allow'.
  warnings.warn(


In [ ]:
grid_cv_results = grid_search.cv_results_
grid_mean_errors = -grid_cv_results['mean_test_score']
grid_running_best = np.minimum.accumulate(grid_mean_errors)

In [ ]:
# TODO: 2. Random Search Implementation
# Use trackio to log the method name, time taken, number of iterations, and best cross-validation score

random_params = {
    'n_estimators': randint(50, 201),
    'max_depth': randint(10, 31),
    'min_samples_split': randint(2, 11)
}

start_time = time.time()
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=random_params,
    n_iter=60,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1,
    random_state=42,
    return_train_score=False
)
random_search.fit(X_train, y_train)
random_time = time.time() - start_time

random_best_params = random_search.best_params_
random_best_score = -random_search.best_score_

print(f'Random Search completed in {random_time:.2f}s')
print(f'Best params: {random_best_params}')
print(f'Best CV MAE: {random_best_score:.4f}')

Fitting 5 folds for each of 60 candidates, totalling 300 fits
Random Search completed in 373.65s
Best params: {'max_depth': 22, 'min_samples_split': 2, 'n_estimators': 138}
Best CV MAE: 13309.7826


In [ ]:
run_random = trackio.init(project='UrbanNest-RentPrediction', name='RandomSearch')
trackio.log({
    'method': 'RandomSearch',
    'best_cv_mae': random_best_score,
    'time_seconds': random_time,
    'n_iterations': 60,
    'best_n_estimators': random_best_params['n_estimators'],
    'best_max_depth': random_best_params['max_depth'],
    'best_min_samples_split': random_best_params['min_samples_split'],
})
trackio.finish()

Exception in thread Thread-259 (_local_batch_sender):
Traceback (most recent call last):
  File "c:\Users\r4849\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "c:\Users\r4849\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\r4849\AppData\Local\Programs\Python\Python313\Lib\site-packages\trackio\run.py", line 158, in _local_batch_sender
    self._write_logs_to_sqlite(logs_to_send)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "c:\Users\r4849\AppData\Local\Programs\Python\Python313\Lib\site-packages\trackio\run.py", line 189, in _write_logs_to_sqlite
    SQLiteStorage.bulk_log(
    ~~~~~~~~~~~~~~~~~~~~~~^
        project=project,
        ^^^^^^^^^^^^^^^^
    ...<4 lines>...
        log_ids=data["log_ids"] if has_log_ids else None,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

* Created new run: RandomSearch
* Run finished. Uploading logs to Trackio (please wait...)


In [ ]:
random_cv_results = random_search.cv_results_
random_mean_errors = -random_cv_results['mean_test_score']
random_running_best = np.minimum.accumulate(random_mean_errors)

In [ ]:
# TODO: 3. Bayesian Optimization (Optuna) Implementation
# Use trackio to log the method name, time taken, number of iterations, and best cross-validation score

optuna_trial_errors = []

def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 10, 30)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42,
        n_jobs=-1
    )
    scores = cross_val_score(model, X_train, y_train, cv=5,
                             scoring='neg_mean_absolute_error')
    mae = -scores.mean()
    optuna_trial_errors.append(mae)
    return mae

start_time = time.time()
study = optuna.create_study(direction='minimize', study_name='RF_BayesianOpt')
study.optimize(objective, n_trials=60, show_progress_bar=True)
optuna_time = time.time() - start_time

optuna_best_params = study.best_params
optuna_best_score = study.best_value

print(f'Bayesian Opt completed in {optuna_time:.2f}s')
print(f'Best params: {optuna_best_params}')
print(f'Best CV MAE: {optuna_best_score:.4f}')

  0%|          | 0/60 [00:00<?, ?it/s]

Bayesian Opt completed in 542.02s
Best params: {'n_estimators': 190, 'max_depth': 26, 'min_samples_split': 2}
Best CV MAE: 13268.6262


In [ ]:
run_optuna = trackio.init(project='UrbanNest-RentPrediction', name='BayesianOptimization')
trackio.log({
    'method': 'BayesianOptimization',
    'best_cv_mae': optuna_best_score,
    'time_seconds': optuna_time,
    'n_iterations': 60,
    'best_n_estimators': optuna_best_params['n_estimators'],
    'best_max_depth': optuna_best_params['max_depth'],
    'best_min_samples_split': optuna_best_params['min_samples_split'],
})
trackio.finish()

* Created new run: BayesianOptimization
* Run finished. Uploading logs to Trackio (please wait...)


In [ ]:
optuna_running_best = np.minimum.accumulate(optuna_trial_errors)

## 3. Evaluation & Plots
Plot the compute trials (iterations) vs. cross-validation error, and plot the hyperparameter space to visualize how the Bayesian method explored the space.

In [ ]:
# TODO: Generate and save trials_vs_error.png
# X-axis: Number of iterations
# Y-axis: Best CV error found so far
# Overlay Grid, Random, and Bayesian methods on the same plot.
plt.figure(figsize=(12, 6))
iterations = range(1, 61)
plt.plot(iterations, grid_running_best, label='Grid Search', linewidth=2, color='#e74c3c')
plt.plot(iterations, random_running_best, label='Random Search', linewidth=2, color='#3498db')
plt.plot(iterations, optuna_running_best, label='Bayesian Optimization', linewidth=2, color='#2ecc71')
plt.xlabel('Number of Iterations / Trials', fontsize=13)
plt.ylabel('Best Mean CV MAE (so far)', fontsize=13)
plt.title('Compute Budget vs. Best Cross-Validation Error', fontsize=15, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/trials_vs_error.png', dpi=150)


In [ ]:
# TODO: Generate and save optuna_hyperparameter_space.png

fig = optuna.visualization.matplotlib.plot_optimization_history(study)
fig.figure.set_size_inches(12, 6)
fig.figure.savefig('plots/optuna_hyperparameter_space.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots/optuna_hyperparameter_space.png')

C:\Users\r4849\AppData\Local\Temp\ipykernel_27024\3200666043.py:3: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  fig = optuna.visualization.matplotlib.plot_optimization_history(study)


Saved plots/optuna_hyperparameter_space.png


C:\Users\r4849\AppData\Local\Temp\ipykernel_27024\3200666043.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Final Testing & Model Saving
Report the best hyperparameters found, train your overall best model on the entire `train.csv`, evaluate on `test.csv`, and save the model file.

In [ ]:
# TODO: Print the best hyperparameters found by all 3 methods
print('=' * 60)
print('BEST HYPERPARAMETERS SUMMARY')
print('=' * 60)
print(f'Grid Search:           {grid_best_params}  (CV MAE: {grid_best_score:.4f})')
print(f'Random Search:         {random_best_params}  (CV MAE: {random_best_score:.4f})')
print(f'Bayesian Optimization: {optuna_best_params}  (CV MAE: {optuna_best_score:.4f})')

results = {
    'GridSearch': (grid_best_params, grid_best_score),
    'RandomSearch': (random_best_params, random_best_score),
    'BayesianOptimization': (optuna_best_params, optuna_best_score),
}
best_method = min(results, key=lambda k: results[k][1])
best_params_overall, best_score_overall = results[best_method]

print(f'\nOverall Best Method: {best_method}')
print(f'Overall Best Params: {best_params_overall}')
print(f'Overall Best CV MAE: {best_score_overall:.4f}')

BEST HYPERPARAMETERS SUMMARY
Grid Search:           {'max_depth': 25, 'min_samples_split': 2, 'n_estimators': 200}  (CV MAE: 13270.7582)
Random Search:         {'max_depth': 22, 'min_samples_split': 2, 'n_estimators': 138}  (CV MAE: 13309.7826)
Bayesian Optimization: {'n_estimators': 190, 'max_depth': 26, 'min_samples_split': 2}  (CV MAE: 13268.6262)

Overall Best Method: BayesianOptimization
Overall Best Params: {'n_estimators': 190, 'max_depth': 26, 'min_samples_split': 2}
Overall Best CV MAE: 13268.6262


In [ ]:
# TODO: Train the best model found on the full X_train
print('Training final model on full training data...')
final_model = RandomForestRegressor(
    n_estimators=best_params_overall['n_estimators'],
    max_depth=best_params_overall['max_depth'],
    min_samples_split=best_params_overall['min_samples_split'],
    random_state=42,
    n_jobs=-1
)
final_model.fit(X_train, y_train)

Training final model on full training data...


,n_estimators,190
,criterion,'squared_error'
,max_depth,26
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
# TODO: Evaluate the model on X_test (Report MAE)
y_pred = final_model.predict(X_test)
test_mae = mean_absolute_error(y_test, y_pred)
print(f'Final Model Test MAE: {test_mae:.4f}')

Final Model Test MAE: 12419.5017


In [ ]:
# Log final results to trackio
run_final = trackio.init(project='UrbanNest-RentPrediction', name='FinalModel')
trackio.log({
    'method': best_method,
    'test_mae': test_mae,
    'best_cv_mae': best_score_overall,
    'n_estimators': best_params_overall['n_estimators'],
    'max_depth': best_params_overall['max_depth'],
    'min_samples_split': best_params_overall['min_samples_split'],
})
trackio.finish()

Final Model Test MAE: 12419.5017
* Created new run: calm-river-2
* Run finished. Uploading logs to Trackio (please wait...)


c:\Users\r4849\AppData\Local\Programs\Python\Python313\Lib\site-packages\trackio\__init__.py:257: UserWarning: * Warning: resume='never' but a run 'FinalModel' already exists in project 'UrbanNest-RentPrediction'. Generating a new name and instead. If you want to resume this run, call init() with resume='must' or resume='allow'.
  warnings.warn(


In [ ]:
# TODO: Save best_model.pkl and any necessary encoders to the models/ folder

model_data = {
    'model': final_model,
    'label_encoders': label_encoders,
    'feature_cols': feature_cols,
    'categorical_cols': categorical_cols,
}

with open('models/best_rf_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print('Saved models/best_rf_model.pkl (model + label encoders + feature info)')

Saved models/best_rf_model.pkl (model + label encoders + feature info)
